# Master Runner: Execute All Notebooks in Order

This notebook runs every project notebook in dependency order and prints diagnostics at each step.

**Execution order:**

| # | Notebook | Phase | Depends On | Output Dir |
|---|----------|-------|------------|------------|
| 01 | `01_state_population_demographics` | Data Collection | — | `nb01_census_population/` |
| 02 | `02_provider_landscape` | Data Collection | NB01 | `nb02_provider_landscape/` |
| 02b | `02b_provider_practice_settings` | Data Collection | NB07 (for state list) | `nb02b_provider_practice_settings/` |
| 03 | `03_payer_mix_insurance` | Data Collection | NB01 | `nb03_payer_mix/` |
| 04 | `04_mental_health_demand` | Data Collection | NB01 | `nb04_mental_health_demand/` |
| 05 | `05_competitive_landscape` | State Analysis | NB01 | `nb05_competitive_landscape/` |
| 06 | `06_regulatory_reimbursement` | State Analysis | NB01 | `nb06_regulatory_reimbursement/` |
| 07 | `07_expansion_opportunity_scoring` | Expansion Modeling | NB01–06 | `nb07_expansion_scoring/` |
| 08 | `08_covered_lives_projections` | Expansion Modeling | NB03, NB07 | `nb08_projections/` |
| 09 | `09_visualizations_recommendations` | Visualizations | NB07, NB08 | `nb09_visualizations/`, `blog_charts/` |
| 10 | `10_revenue_unit_economics` | Expansion Modeling | NB07, NB08 | `nb10_revenue_unit_economics/` |

**Note:** NB02b (Provider Practice Settings) requires the BLS OES download file. If it hasn't been downloaded yet, that notebook will attempt to fetch it. Run it after NB07 since it needs the expansion scores for state classification.

**Usage:** Run all cells, or run selectively by setting `SKIP_NOTEBOOKS` below.

In [1]:
import subprocess
import sys
import os
import time
import json
from datetime import datetime
from pathlib import Path

# --- Configuration ---
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))  # This notebook sits at project root
NOTEBOOKS_DIR = os.path.join(PROJECT_ROOT, 'notebooks')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs')

# Set to a list of notebook keys to skip (e.g., ['02b', '10'])
# Or set to [] to run everything
SKIP_NOTEBOOKS = []

# Timeout per notebook in seconds (default: 10 minutes)
TIMEOUT_SECONDS = 600

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebooks dir: {NOTEBOOKS_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Skip list: {SKIP_NOTEBOOKS or 'None (running all)'}")
print(f"Timeout: {TIMEOUT_SECONDS}s per notebook")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Project root: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/octave_behavioral_health_expansion
Notebooks dir: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/octave_behavioral_health_expansion/notebooks
Output dir: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/octave_behavioral_health_expansion/data/outputs
Skip list: None (running all)
Timeout: 600s per notebook
Started at: 2026-03-30 20:04:40


In [2]:
# --- Define notebook execution order ---

NOTEBOOK_PIPELINE = [
    {
        'key': '01',
        'name': 'State Population & Demographics',
        'path': '1_data_collection/01_state_population_demographics.ipynb',
        'output_dir': 'nb01_census_population',
        'expected_outputs': ['state_demographics.csv'],
        'depends_on': [],
    },
    {
        'key': '02',
        'name': 'Provider Landscape',
        'path': '1_data_collection/02_provider_landscape.ipynb',
        'output_dir': 'nb02_provider_landscape',
        'expected_outputs': ['state_provider_counts.csv'],
        'depends_on': ['01'],
    },
    {
        'key': '03',
        'name': 'Payer Mix & Insurance',
        'path': '1_data_collection/03_payer_mix_insurance.ipynb',
        'output_dir': 'nb03_payer_mix',
        'expected_outputs': ['state_payer_mix.csv', 'state_octave_payer_presence.csv', 'state_octave_covered_lives.csv'],
        'depends_on': ['01'],
    },
    {
        'key': '04',
        'name': 'Mental Health Demand',
        'path': '1_data_collection/04_mental_health_demand.ipynb',
        'output_dir': 'nb04_mental_health_demand',
        'expected_outputs': ['state_mental_health_demand.csv'],
        'depends_on': ['01'],
    },
    {
        'key': '05',
        'name': 'Competitive Landscape',
        'path': '2_state_analysis/05_competitive_landscape.ipynb',
        'output_dir': 'nb05_competitive_landscape',
        'expected_outputs': ['state_competitive_landscape.csv'],
        'depends_on': ['01'],
    },
    {
        'key': '06',
        'name': 'Regulatory & Reimbursement',
        'path': '2_state_analysis/06_regulatory_reimbursement.ipynb',
        'output_dir': 'nb06_regulatory_reimbursement',
        'expected_outputs': ['state_regulatory_reimbursement.csv'],
        'depends_on': ['01'],
    },
    {
        'key': '07',
        'name': 'Expansion Opportunity Scoring',
        'path': '3_expansion_modeling/07_expansion_opportunity_scoring.ipynb',
        'output_dir': 'nb07_expansion_scoring',
        'expected_outputs': ['state_expansion_scores.csv'],
        'depends_on': ['01', '02', '03', '04', '05', '06'],
    },
    {
        'key': '02b',
        'name': 'Provider Practice Settings',
        'path': '1_data_collection/02b_provider_practice_settings.ipynb',
        'output_dir': 'nb02b_provider_practice_settings',
        'expected_outputs': ['state_provider_practice_settings.csv'],
        'depends_on': ['07'],
    },
    {
        'key': '08',
        'name': 'Covered Lives Projections',
        'path': '3_expansion_modeling/08_covered_lives_projections.ipynb',
        'output_dir': 'nb08_projections',
        'expected_outputs': [],  # Check after first run
        'depends_on': ['03', '07'],
    },
    {
        'key': '09',
        'name': 'Visualizations & Recommendations',
        'path': '4_visualizations_recommendations/09_visualizations_recommendations.ipynb',
        'output_dir': 'nb09_visualizations',
        'expected_outputs': [],  # Generates PNGs and blog charts
        'depends_on': ['07', '08'],
    },
    {
        'key': '10',
        'name': 'Revenue & Unit Economics',
        'path': '3_expansion_modeling/10_revenue_unit_economics.ipynb',
        'output_dir': 'nb10_revenue_unit_economics',
        'expected_outputs': [],
        'depends_on': ['07', '08'],
    },
]

print(f"Pipeline: {len(NOTEBOOK_PIPELINE)} notebooks")
for nb in NOTEBOOK_PIPELINE:
    skip_marker = ' [SKIP]' if nb['key'] in SKIP_NOTEBOOKS else ''
    print(f"  {nb['key']:>3s}. {nb['name']:<40s} depends on: {nb['depends_on'] or '—'}{skip_marker}")

Pipeline: 11 notebooks
   01. State Population & Demographics          depends on: —
   02. Provider Landscape                       depends on: ['01']
   03. Payer Mix & Insurance                    depends on: ['01']
   04. Mental Health Demand                     depends on: ['01']
   05. Competitive Landscape                    depends on: ['01']
   06. Regulatory & Reimbursement               depends on: ['01']
   07. Expansion Opportunity Scoring            depends on: ['01', '02', '03', '04', '05', '06']
  02b. Provider Practice Settings               depends on: ['07']
   08. Covered Lives Projections                depends on: ['03', '07']
   09. Visualizations & Recommendations         depends on: ['07', '08']
   10. Revenue & Unit Economics                 depends on: ['07', '08']


In [3]:
# --- Helper functions ---

def check_outputs(output_dir_name, expected_files):
    """Check if expected output files exist and report their sizes."""
    out_path = os.path.join(OUTPUT_DIR, output_dir_name)
    results = []
    
    if not os.path.exists(out_path):
        return [{'file': f, 'exists': False, 'size': 0} for f in expected_files]
    
    # Check expected files
    for f in expected_files:
        fpath = os.path.join(out_path, f)
        if os.path.exists(fpath):
            size = os.path.getsize(fpath)
            results.append({'file': f, 'exists': True, 'size': size})
        else:
            results.append({'file': f, 'exists': False, 'size': 0})
    
    # Also list any other files in the directory
    all_files = os.listdir(out_path)
    extra_files = [f for f in all_files if f not in expected_files]
    for f in extra_files:
        fpath = os.path.join(out_path, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath)
            results.append({'file': f, 'exists': True, 'size': size, 'extra': True})
    
    return results


def run_notebook(nb_path, timeout=600):
    """Execute a notebook using nbconvert and return result dict."""
    full_path = os.path.join(NOTEBOOKS_DIR, nb_path)
    
    if not os.path.exists(full_path):
        return {
            'success': False,
            'error': f'Notebook not found: {full_path}',
            'duration': 0,
            'stdout': '',
            'stderr': '',
        }
    
    # Get the notebook's directory for proper working directory
    nb_dir = os.path.dirname(full_path)
    
    start = time.time()
    try:
        result = subprocess.run(
            [
                sys.executable, '-m', 'jupyter', 'nbconvert',
                '--to', 'notebook',
                '--execute',
                '--inplace',
                '--ExecutePreprocessor.timeout=' + str(timeout),
                full_path
            ],
            capture_output=True,
            text=True,
            timeout=timeout + 30,  # Extra buffer for nbconvert overhead
            cwd=nb_dir,  # Run from the notebook's directory
        )
        duration = time.time() - start
        
        return {
            'success': result.returncode == 0,
            'error': result.stderr if result.returncode != 0 else '',
            'duration': duration,
            'stdout': result.stdout,
            'stderr': result.stderr,
            'returncode': result.returncode,
        }
    except subprocess.TimeoutExpired:
        duration = time.time() - start
        return {
            'success': False,
            'error': f'TIMEOUT after {timeout}s',
            'duration': duration,
            'stdout': '',
            'stderr': '',
        }
    except Exception as e:
        duration = time.time() - start
        return {
            'success': False,
            'error': str(e),
            'duration': duration,
            'stdout': '',
            'stderr': '',
        }


def extract_cell_error(nb_path):
    """If a notebook failed, find the cell that raised the error."""
    full_path = os.path.join(NOTEBOOKS_DIR, nb_path)
    try:
        with open(full_path, 'r') as f:
            nb = json.load(f)
        
        for i, cell in enumerate(nb.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            outputs = cell.get('outputs', [])
            for output in outputs:
                if output.get('output_type') == 'error':
                    ename = output.get('ename', 'Unknown')
                    evalue = output.get('evalue', '')
                    traceback_lines = output.get('traceback', [])
                    # Get last few lines of traceback (strip ANSI codes)
                    import re
                    clean_tb = [re.sub(r'\x1b\[[0-9;]*m', '', line) for line in traceback_lines[-3:]]
                    source_preview = ''.join(cell.get('source', []))[:200]
                    return {
                        'cell_index': i,
                        'error_type': ename,
                        'error_message': evalue,
                        'traceback': '\n'.join(clean_tb),
                        'cell_source_preview': source_preview,
                    }
    except Exception:
        pass
    return None


print("Helper functions loaded.")

Helper functions loaded.


In [4]:
# --- Pre-flight check: verify all notebooks exist ---

print("Pre-flight checks:")
print("=" * 70)
all_exist = True

for nb in NOTEBOOK_PIPELINE:
    full_path = os.path.join(NOTEBOOKS_DIR, nb['path'])
    exists = os.path.exists(full_path)
    size = os.path.getsize(full_path) if exists else 0
    status = f'OK ({size/1024:.0f} KB)' if exists else 'MISSING'
    marker = '  ✓' if exists else '  ✗'
    print(f"{marker} NB{nb['key']:>3s}: {nb['path']:<60s} {status}")
    if not exists:
        all_exist = False

# Check nbconvert is available
try:
    result = subprocess.run([sys.executable, '-m', 'jupyter', 'nbconvert', '--version'],
                          capture_output=True, text=True)
    nbconvert_version = result.stdout.strip()
    print(f"\n  ✓ nbconvert version: {nbconvert_version}")
except Exception as e:
    print(f"\n  ✗ nbconvert not available: {e}")
    print("    Install with: pip install nbconvert")

print(f"\nPython: {sys.version}")
print(f"\nAll notebooks found: {'YES' if all_exist else 'NO — fix missing notebooks before running'}")

Pre-flight checks:
  ✓ NB 01: 1_data_collection/01_state_population_demographics.ipynb     OK (96 KB)
  ✓ NB 02: 1_data_collection/02_provider_landscape.ipynb                OK (210 KB)
  ✓ NB 03: 1_data_collection/03_payer_mix_insurance.ipynb               OK (208 KB)
  ✓ NB 04: 1_data_collection/04_mental_health_demand.ipynb              OK (267 KB)
  ✓ NB 05: 2_state_analysis/05_competitive_landscape.ipynb              OK (375 KB)
  ✓ NB 06: 2_state_analysis/06_regulatory_reimbursement.ipynb           OK (299 KB)
  ✓ NB 07: 3_expansion_modeling/07_expansion_opportunity_scoring.ipynb  OK (690 KB)
  ✓ NB02b: 1_data_collection/02b_provider_practice_settings.ipynb       OK (27 KB)
  ✓ NB 08: 3_expansion_modeling/08_covered_lives_projections.ipynb      OK (395 KB)
  ✓ NB 09: 4_visualizations_recommendations/09_visualizations_recommendations.ipynb OK (507 KB)
  ✓ NB 10: 3_expansion_modeling/10_revenue_unit_economics.ipynb         OK (116 KB)

  ✓ nbconvert version: 7.17.0

Python: 3.9.5 (

In [5]:
# --- Execute all notebooks ---

run_results = {}
pipeline_start = time.time()

print("=" * 70)
print(f"EXECUTING PIPELINE — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

for nb in NOTEBOOK_PIPELINE:
    key = nb['key']
    
    # Skip if in skip list
    if key in SKIP_NOTEBOOKS:
        print(f"\n{'─'*70}")
        print(f"⏭  NB{key}: {nb['name']} — SKIPPED (in SKIP_NOTEBOOKS)")
        run_results[key] = {'success': None, 'skipped': True, 'duration': 0}
        continue
    
    # Check if dependencies succeeded
    dep_failed = [d for d in nb['depends_on'] if d in run_results and run_results[d].get('success') == False]
    if dep_failed:
        print(f"\n{'─'*70}")
        print(f"⏭  NB{key}: {nb['name']} — SKIPPED (dependency failed: NB{', NB'.join(dep_failed)})")
        run_results[key] = {'success': None, 'skipped': True, 'duration': 0, 'reason': f'Dependency failed: {dep_failed}'}
        continue
    
    print(f"\n{'─'*70}")
    print(f"▶  NB{key}: {nb['name']}")
    print(f"   Path: notebooks/{nb['path']}")
    print(f"   Started: {datetime.now().strftime('%H:%M:%S')}")
    
    # Run the notebook
    result = run_notebook(nb['path'], timeout=TIMEOUT_SECONDS)
    run_results[key] = result
    
    if result['success']:
        print(f"   ✅ PASSED in {result['duration']:.1f}s")
        
        # Check outputs
        outputs = check_outputs(nb['output_dir'], nb['expected_outputs'])
        if outputs:
            print(f"   Outputs ({nb['output_dir']}/):")
            for o in outputs:
                size_str = f"{o['size']/1024:.1f} KB" if o['size'] > 0 else 'EMPTY'
                extra = ' (extra)' if o.get('extra') else ''
                marker = '✓' if o['exists'] else '✗'
                print(f"     {marker} {o['file']:<45s} {size_str}{extra}")
    else:
        print(f"   ❌ FAILED after {result['duration']:.1f}s")
        
        # Show error summary
        if result.get('error'):
            # Truncate long error messages
            error_lines = result['error'].strip().split('\n')
            print(f"   Error summary:")
            for line in error_lines[-10:]:  # Last 10 lines
                print(f"     {line}")
        
        # Try to extract the specific cell that failed
        cell_error = extract_cell_error(nb['path'])
        if cell_error:
            print(f"\n   Failed cell (index {cell_error['cell_index']}):")
            print(f"   Error type: {cell_error['error_type']}")
            print(f"   Message: {cell_error['error_message']}")
            print(f"   Cell code preview:")
            for line in cell_error['cell_source_preview'].split('\n')[:5]:
                print(f"     | {line}")
            print(f"   Traceback (last 3 lines):")
            for line in cell_error['traceback'].split('\n')[-3:]:
                print(f"     {line}")

pipeline_duration = time.time() - pipeline_start
print(f"\n{'='*70}")
print(f"Pipeline finished in {pipeline_duration:.1f}s ({pipeline_duration/60:.1f} min)")
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

EXECUTING PIPELINE — 2026-03-30 20:04:43

──────────────────────────────────────────────────────────────────────
▶  NB01: State Population & Demographics
   Path: notebooks/1_data_collection/01_state_population_demographics.ipynb
   Started: 20:04:43
   ✅ PASSED in 8.0s
   Outputs (nb01_census_population/):
     ✗ state_demographics.csv                        EMPTY
     ✓ state_population_base.csv                     2.6 KB (extra)
     ✓ state_population_overview.png                 79.0 KB (extra)

──────────────────────────────────────────────────────────────────────
▶  NB02: Provider Landscape
   Path: notebooks/1_data_collection/02_provider_landscape.ipynb
   Started: 20:04:51
   ✅ PASSED in 125.0s
   Outputs (nb02_provider_landscape/):
     ✓ state_provider_counts.csv                     8.6 KB
     ✓ provider_supply_by_state.png                  88.5 KB (extra)
     ✓ octave_vs_expansion_provider_supply.png       87.3 KB (extra)

─────────────────────────────────────────────────

In [6]:
# --- Final Summary Report ---

print("\n" + "=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)

passed = []
failed = []
skipped = []

for nb in NOTEBOOK_PIPELINE:
    key = nb['key']
    r = run_results.get(key, {})
    
    if r.get('skipped'):
        status = '⏭  SKIP'
        reason = r.get('reason', 'User skip list')
        skipped.append(key)
        print(f"  {status}  NB{key:>3s}  {nb['name']:<40s}  ({reason})")
    elif r.get('success'):
        status = '✅ PASS'
        passed.append(key)
        print(f"  {status}  NB{key:>3s}  {nb['name']:<40s}  {r['duration']:.1f}s")
    elif r.get('success') == False:
        status = '❌ FAIL'
        failed.append(key)
        print(f"  {status}  NB{key:>3s}  {nb['name']:<40s}  {r['duration']:.1f}s")
    else:
        status = '❓ N/A '
        print(f"  {status}  NB{key:>3s}  {nb['name']:<40s}")

print(f"\n{'─'*70}")
print(f"  Passed: {len(passed)}  |  Failed: {len(failed)}  |  Skipped: {len(skipped)}  |  Total: {len(NOTEBOOK_PIPELINE)}")
print(f"  Total time: {pipeline_duration:.1f}s ({pipeline_duration/60:.1f} min)")

if failed:
    print(f"\n⚠️  FAILED NOTEBOOKS: NB{', NB'.join(failed)}")
    print(f"   Review the error diagnostics above to identify the issue.")
    print(f"   Common causes:")
    print(f"     - Missing pip packages (run: pip install <package>)")
    print(f"     - Missing input CSV from a prior notebook")
    print(f"     - API rate limit or network timeout (BLS, Census)")
    print(f"     - Column name changes in upstream data")
else:
    print(f"\n🎉 All notebooks passed!")


PIPELINE SUMMARY
  ✅ PASS  NB 01  State Population & Demographics           8.0s
  ✅ PASS  NB 02  Provider Landscape                        125.0s
  ✅ PASS  NB 03  Payer Mix & Insurance                     10.3s
  ✅ PASS  NB 04  Mental Health Demand                      10.1s
  ✅ PASS  NB 05  Competitive Landscape                     11.8s
  ✅ PASS  NB 06  Regulatory & Reimbursement                11.0s
  ✅ PASS  NB 07  Expansion Opportunity Scoring             13.7s
  ✅ PASS  NB02b  Provider Practice Settings                34.0s
  ✅ PASS  NB 08  Covered Lives Projections                 12.2s
  ✅ PASS  NB 09  Visualizations & Recommendations          12.6s
  ✅ PASS  NB 10  Revenue & Unit Economics                  9.5s

──────────────────────────────────────────────────────────────────────
  Passed: 11  |  Failed: 0  |  Skipped: 0  |  Total: 11
  Total time: 258.3s (4.3 min)

🎉 All notebooks passed!


In [7]:
# --- Output file inventory ---

print("\n" + "=" * 70)
print("OUTPUT FILE INVENTORY")
print("=" * 70)

total_files = 0
total_size = 0

for dirname in sorted(os.listdir(OUTPUT_DIR)):
    dirpath = os.path.join(OUTPUT_DIR, dirname)
    if not os.path.isdir(dirpath):
        continue
    
    files = [f for f in os.listdir(dirpath) if os.path.isfile(os.path.join(dirpath, f))]
    dir_size = sum(os.path.getsize(os.path.join(dirpath, f)) for f in files)
    
    print(f"\n  📁 {dirname}/ ({len(files)} files, {dir_size/1024:.1f} KB)")
    for f in sorted(files):
        fpath = os.path.join(dirpath, f)
        fsize = os.path.getsize(fpath)
        mtime = datetime.fromtimestamp(os.path.getmtime(fpath)).strftime('%H:%M:%S')
        print(f"     {f:<50s} {fsize/1024:>8.1f} KB  ({mtime})")
        total_files += 1
        total_size += fsize

print(f"\n{'─'*70}")
print(f"  Total: {total_files} files, {total_size/1024/1024:.1f} MB")


OUTPUT FILE INVENTORY

  📁 blog_charts/ (20 files, 307.3 KB)
     chart_benchmark_comparison.html                         3.2 KB  (18:33:47)
     chart_competitive_map.html                              2.3 KB  (15:50:58)
     chart_covered_lives_comparison.html                     2.5 KB  (14:30:56)
     chart_demand_supply_scatter.html                        5.4 KB  (14:30:56)
     chart_footprint_map.html                                8.3 KB  (16:10:34)
     chart_growth_curves.html                                2.6 KB  (18:04:28)
     chart_heatmap_expansion.html                            4.9 KB  (17:35:10)
     chart_opportunity_map.html                             15.8 KB  (17:03:49)
     chart_practice_settings.html                            3.0 KB  (19:33:10)
     chart_practice_settings_comparison.html                 3.1 KB  (19:50:42)
     chart_provider_density_map.html                         2.5 KB  (15:33:37)
     chart_provider_types_bar.html                        